# Agrupamento (Clustering) com o Dataset Iris

Neste notebook vamos aprender clustering na prática usando o **Iris dataset**.

## A diferença fundamental

> No clustering, **fingimos que não conhecemos os rótulos** das flores.
> O desafio é: o algoritmo consegue descobrir as 3 espécies sozinho, apenas pelas medidas?

**Features disponíveis:** `sepal length`, `sepal width`, `petal length`, `petal width`  
**Rótulos (escondidos do modelo):** `setosa`, `versicolor`, `virginica`

## Algoritmos que vamos comparar
1. **K-Means** — particiona em k clusters minimizando inércia
2. **DBSCAN** — agrupa por densidade, detecta outliers automaticamente
3. **Hierárquico (Ward)** — constrói hierarquia de clusters, visualizado como dendrograma


---
## 1. Importações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Bibliotecas carregadas com sucesso!')

---
## 2. Carregando os Dados (sem rótulos)

In [ ]:
iris = load_iris()

# X = apenas as features — sem rótulos!
X = iris.data
y_true = iris.target  # guardado para comparação posterior, não usado no treino

df = pd.DataFrame(X, columns=iris.feature_names)

print('=== Dados carregados (modo não supervisionado) ===')
print(f'Amostras: {X.shape[0]} | Features: {X.shape[1]}')
print(f'Features: {list(iris.feature_names)}')
print(f'\n[!] Rótulos NÃO fornecidos ao modelo — ele deve descobrir os grupos sozinho')
print('\nPrimeiras 5 linhas:')
df.head()

In [ ]:
print('=== Estatísticas Descritivas ===')
df.describe().round(2)

---
## 3. Visualização Exploratória (EDA)

In [ ]:
# Scatter plots das combinações de features — sem cor de espécie!
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Iris Dataset — Exploração SEM rótulos\n(como o modelo enxerga os dados)', fontsize=14, fontweight='bold')

feature_pairs = [
    (0, 1), (0, 2), (0, 3),
    (1, 2), (1, 3), (2, 3)
]

for ax, (i, j) in zip(axes.flatten(), feature_pairs):
    ax.scatter(X[:, i], X[:, j], alpha=0.5, color='steelblue', s=40)
    ax.set_xlabel(iris.feature_names[i], fontsize=9)
    ax.set_ylabel(iris.feature_names[j], fontsize=9)
    ax.set_title(f'{iris.feature_names[i][:10]} × {iris.feature_names[j][:10]}', fontsize=10)

plt.tight_layout()
plt.show()

print('Observe: mesmo sem rótulos, é possível notar grupos naturais nos dados!')

In [ ]:
# Matriz de correlação
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Matriz de Correlação', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Pré-processamento — Normalização

Clustering usa **distâncias** entre pontos. Se uma feature tem escala muito maior que outra, ela domina o cálculo de distância. A normalização é **obrigatória** para K-Means e DBSCAN.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('=== Normalização (StandardScaler) ===')
print(f'Antes — média: {X.mean(axis=0).round(2)}')
print(f'Antes — desvio: {X.std(axis=0).round(2)}')
print(f'\nDepois — média: {X_scaled.mean(axis=0).round(4)}')
print(f'Depois — desvio: {X_scaled.std(axis=0).round(4)}')
print('\nAgora todas as features estão na mesma escala!')

---
## 5. K-Means

### 5.1 Escolhendo o k — Método do Cotovelo + Silhouette

In [ ]:
inercias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means — Escolha do Número de Clusters', fontsize=14, fontweight='bold')

# Elbow Method
axes[0].plot(k_range, inercias, 'o-', color='steelblue', lw=2, markersize=7)
axes[0].axvline(3, color='red', linestyle='--', lw=2, label='k=3 (cotovelo)')
axes[0].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inércia (WCSS)', fontsize=12)
axes[0].set_title('Método do Cotovelo (Elbow)', fontsize=12)
axes[0].legend()
axes[0].set_xticks(list(k_range))

# Silhouette Score
axes[1].plot(k_range, silhouettes, 's-', color='darkorange', lw=2, markersize=7)
best_k_sil = k_range[np.argmax(silhouettes)]
axes[1].axvline(best_k_sil, color='red', linestyle='--', lw=2,
                label=f'Melhor k={best_k_sil} (silhouette={max(silhouettes):.3f})')
axes[1].set_xlabel('Número de Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score por k', fontsize=12)
axes[1].legend()
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.show()

print(f'Cotovelo sugere: k=3')
print(f'Silhouette sugere: k={best_k_sil} (score={max(silhouettes):.4f})')
print(f'→ Usaremos k=3 (sabemos que o dataset tem 3 espécies)')

### 5.2 Treinando o K-Means com k=3

In [ ]:
km = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
km.fit(X_scaled)

labels_km = km.labels_
centroides = km.cluster_centers_

print('=== K-Means (k=3) ===')
print(f'Inércia total: {km.inertia_:.4f}')
print(f'Iterações até convergência: {km.n_iter_}')
print(f'\nTamanho dos clusters:')
for i, count in enumerate(np.bincount(labels_km)):
    print(f'  Cluster {i}: {count} amostras')

print(f'\nCentroides (espaço normalizado):')
cent_df = pd.DataFrame(centroides, columns=iris.feature_names)
cent_df.index = [f'Centroide {i}' for i in range(3)]
cent_df.round(3)

---
## 6. DBSCAN

### 6.1 Calibrando ε — k-Distance Plot

In [ ]:
# k-distance plot: distância ao k-ésimo vizinho mais próximo (k=5)
from sklearn.neighbors import NearestNeighbors

k_neighbors = 5
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
dist_k = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(dist_k, color='steelblue', lw=2)
plt.axhline(0.5, color='red', linestyle='--', lw=2, label='ε = 0.5 (sugestão)')
plt.xlabel('Ponto (ordenado por distância decrescente)', fontsize=11)
plt.ylabel(f'Distância ao {k_neighbors}º vizinho', fontsize=11)
plt.title(f'k-Distance Plot (k={k_neighbors}) — Calibração do ε para DBSCAN', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print('O "cotovelo" do gráfico indica o valor de ε adequado.')
print('Valores acima do cotovelo → outliers | abaixo → pontos em regiões densas')

### 6.2 Treinando o DBSCAN

In [ ]:
db = DBSCAN(eps=0.5, min_samples=5)
labels_db = db.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = np.sum(labels_db == -1)

print('=== DBSCAN (ε=0.5, min_samples=5) ===')
print(f'Clusters encontrados: {n_clusters_db}')
print(f'Outliers (ruído):     {n_noise_db} amostras ({n_noise_db/len(X)*100:.1f}%)')
print(f'\nDistribuição:')
for label in sorted(set(labels_db)):
    nome = 'Ruído (outliers)' if label == -1 else f'Cluster {label}'
    count = np.sum(labels_db == label)
    print(f'  {nome}: {count} amostras')

---
## 7. Agrupamento Hierárquico

### 7.1 Dendrograma

In [ ]:
# Calcular a hierarquia de ligações (Ward)
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(16, 6))
dendrogram(
    Z,
    truncate_mode='lastp',  # mostrar apenas os últimos p clusters mesclados
    p=20,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True,
    color_threshold=6.0
)
plt.axhline(6.0, color='red', linestyle='--', lw=2, label='Corte → 3 clusters')
plt.xlabel('Amostras (agrupadas)', fontsize=11)
plt.ylabel('Distância (Ward)', fontsize=11)
plt.title('Dendrograma — Agrupamento Hierárquico (Ward)', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print('A linha vermelha (corte) determina quantos clusters extraímos.')
print('3 ramos abaixo do corte → 3 clusters.')

### 7.2 Treinando o Agrupamento Hierárquico

In [ ]:
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hc = hc.fit_predict(X_scaled)

print('=== Agrupamento Hierárquico (Ward, k=3) ===')
print(f'Clusters encontrados: {hc.n_clusters_}')
print(f'\nTamanho dos clusters:')
for i, count in enumerate(np.bincount(labels_hc)):
    print(f'  Cluster {i}: {count} amostras')

---
## 8. Visualizando os Clusters

Como temos 4 features, usamos **PCA** para reduzir para 2 dimensões e visualizar.

In [ ]:
# Reduzir para 2D com PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

variancia = pca.explained_variance_ratio_
print(f'Variância explicada: PC1={variancia[0]:.2%} | PC2={variancia[1]:.2%} | Total={sum(variancia):.2%}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Clusters Encontrados vs Rótulos Reais (PCA 2D)', fontsize=15, fontweight='bold')

# Paletas
paleta_3 = ['#e74c3c', '#2ecc71', '#3498db']
paleta_db = {-1: '#aaaaaa', 0: '#e74c3c', 1: '#2ecc71', 2: '#3498db', 3: '#f39c12'}

# Plot 1: Rótulos reais (gabarito)
ax = axes[0][0]
for cls, color, nome in zip([0, 1, 2], paleta_3, iris.target_names):
    mask = y_true == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color, alpha=0.7, s=50, label=nome)
ax.set_title('Rótulos Reais (gabarito)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({variancia[0]:.1%})', fontsize=10)
ax.set_ylabel(f'PC2 ({variancia[1]:.1%})', fontsize=10)
ax.legend(fontsize=9)

# Plot 2: K-Means
ax = axes[0][1]
for cls, color in enumerate(paleta_3):
    mask = labels_km == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color, alpha=0.7, s=50, label=f'Cluster {cls}')
# Centroides no espaço PCA
cent_pca = pca.transform(centroides)
ax.scatter(cent_pca[:, 0], cent_pca[:, 1], marker='X', s=200,
           color='black', zorder=5, label='Centroides')
ax.set_title('K-Means (k=3)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({variancia[0]:.1%})', fontsize=10)
ax.set_ylabel(f'PC2 ({variancia[1]:.1%})', fontsize=10)
ax.legend(fontsize=9)

# Plot 3: DBSCAN
ax = axes[1][0]
for cls in sorted(set(labels_db)):
    mask = labels_db == cls
    nome = 'Ruído' if cls == -1 else f'Cluster {cls}'
    marker = 'x' if cls == -1 else 'o'
    color = paleta_db.get(cls, '#f39c12')
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color,
               alpha=0.7, s=50, label=nome, marker=marker)
ax.set_title('DBSCAN (ε=0.5, min_samples=5)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({variancia[0]:.1%})', fontsize=10)
ax.set_ylabel(f'PC2 ({variancia[1]:.1%})', fontsize=10)
ax.legend(fontsize=9)

# Plot 4: Hierárquico
ax = axes[1][1]
for cls, color in enumerate(paleta_3):
    mask = labels_hc == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color, alpha=0.7, s=50, label=f'Cluster {cls}')
ax.set_title('Hierárquico Ward (k=3)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({variancia[0]:.1%})', fontsize=10)
ax.set_ylabel(f'PC2 ({variancia[1]:.1%})', fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 9. Avaliação com Métricas Internas

Como não usamos rótulos no treino, avaliamos com métricas que medem **coesão** e **separação** dos clusters.

In [ ]:
# Filtrar amostras de ruído do DBSCAN para métricas
mask_db = labels_db != -1
X_db_valid = X_scaled[mask_db]
labels_db_valid = labels_db[mask_db]

print('=== Métricas Internas de Clustering ===')
print(f'\n  {"Algoritmo":25s} | {"Silhouette":>12} | {"Davies-Bouldin":>16} | {"Calinski-Harabász":>18}')
print(f'  {"-"*25}-+-{"-"*12}-+-{"-"*16}-+-{"-"*18}')
print(f'  {"(maior melhor)":25s} | {"(maior melhor)":>12} | {"(menor melhor)":>16} | {"(maior melhor)":>18}')
print(f'  {"-"*25}-+-{"-"*12}-+-{"-"*16}-+-{"-"*18}')

resultados_metricas = {}

for nome, X_eval, labels_eval in [
    ('K-Means', X_scaled, labels_km),
    ('DBSCAN (sem ruído)', X_db_valid, labels_db_valid),
    ('Hierárquico', X_scaled, labels_hc),
]:
    n_unique = len(set(labels_eval))
    if n_unique < 2:
        print(f'  {nome:25s} | {"N/A":>12} | {"N/A":>16} | {"N/A":>18}')
        continue
    sil = silhouette_score(X_eval, labels_eval)
    db_idx = davies_bouldin_score(X_eval, labels_eval)
    ch = calinski_harabasz_score(X_eval, labels_eval)
    resultados_metricas[nome] = {'silhouette': sil, 'davies_bouldin': db_idx, 'calinski': ch}
    print(f'  {nome:25s} | {sil:>12.4f} | {db_idx:>16.4f} | {ch:>18.2f}')

---
## 10. Comparação com os Rótulos Reais (ARI)

O **Adjusted Rand Index (ARI)** mede o quanto os clusters encontrados concordam com os rótulos reais, corrigindo para concordâncias aleatórias.

- `ARI = 1.0` → concordância perfeita com os rótulos reais
- `ARI = 0.0` → concordância aleatória
- `ARI < 0` → pior que aleatório

In [ ]:
ari_km = adjusted_rand_score(y_true, labels_km)
ari_db = adjusted_rand_score(y_true[mask_db], labels_db_valid)
ari_hc = adjusted_rand_score(y_true, labels_hc)

print('=== Adjusted Rand Index — Concordância com Rótulos Reais ===')
print(f'  (Esta métrica usa os rótulos reais, apenas para avaliação final)')
print(f'\n  K-Means:     {ari_km:.4f}')
print(f'  DBSCAN:      {ari_db:.4f}  (calculado apenas sobre os {mask_db.sum()} pontos não-ruído)')
print(f'  Hierárquico: {ari_hc:.4f}')

melhor = max([('K-Means', ari_km), ('DBSCAN', ari_db), ('Hierárquico', ari_hc)], key=lambda x: x[1])
print(f'\n  Melhor ARI: {melhor[0]} ({melhor[1]:.4f})')

In [ ]:
# Visualizar concordância: K-Means vs Rótulos Reais
from sklearn.metrics import confusion_matrix
import itertools

# Encontrar o mapeamento ótimo de cluster → espécie
def melhor_mapeamento(y_true, labels):
    """Permuta os rótulos do cluster para maximizar concordância com y_true."""
    from itertools import permutations
    n_cls = len(set(labels))
    melhor_acc = 0
    melhor_perm = None
    for perm in permutations(range(n_cls)):
        mapa = {old: new for old, new in enumerate(perm)}
        remapped = np.array([mapa[l] for l in labels])
        acc = np.mean(remapped == y_true)
        if acc > melhor_acc:
            melhor_acc = acc
            melhor_perm = remapped
    return melhor_perm, melhor_acc

labels_km_mapped, acc_km = melhor_mapeamento(y_true, labels_km)
labels_hc_mapped, acc_hc = melhor_mapeamento(y_true, labels_hc)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Concordância dos Clusters com as Espécies Reais', fontsize=14, fontweight='bold')

from sklearn.metrics import ConfusionMatrixDisplay
for ax, labels_mapped, acc, titulo in [
    (axes[0], labels_km_mapped, acc_km, f'K-Means — Accuracy mapeada: {acc_km:.2%}'),
    (axes[1], labels_hc_mapped, acc_hc, f'Hierárquico — Accuracy mapeada: {acc_hc:.2%}'),
]:
    cm = confusion_matrix(y_true, labels_mapped)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Cluster (mapeado)', fontsize=10)
    ax.set_ylabel('Espécie Real', fontsize=10)

plt.tight_layout()
plt.show()
print('Diagonal = concordância | Fora da diagonal = discordância')

---
## 11. Predizendo Clusters para Novos Dados

In [ ]:
# Novas flores sem rótulo
# [sepal_length, sepal_width, petal_length, petal_width]
novas_flores = np.array([
    [5.0, 3.4, 1.5, 0.2],  # pequena pétala → provavelmente setosa
    [6.0, 2.8, 4.5, 1.5],  # pétala média → provavelmente versicolor
    [7.0, 3.1, 6.0, 2.0],  # pétala grande → provavelmente virginica
    [5.8, 2.7, 4.1, 1.0],  # caso intermediário
])

novas_flores_scaled = scaler.transform(novas_flores)

# K-Means suporta predict() diretamente
pred_km = km.predict(novas_flores_scaled)

# DBSCAN não tem predict() — calculamos pelo centroide mais próximo do cluster
# Para hierárquico, também não há predict() — usamos o centroide mais próximo
centroides_hc = np.array([
    X_scaled[labels_hc == i].mean(axis=0) for i in range(3)
])
dist_hc = cdist(novas_flores_scaled, centroides_hc)
pred_hc = dist_hc.argmin(axis=1)

print('=== Predição de Cluster para Novas Flores ===')
print(f'\n  {"Flor":>4} | {"sepal_l":>7} | {"sepal_w":>7} | {"petal_l":>7} | {"petal_w":>7} | {"K-Means":>9} | {"Hierárq.":>9}')
print(f'  {"-"*4}-+-{"-"*7}-+-{"-"*7}-+-{"-"*7}-+-{"-"*7}-+-{"-"*9}-+-{"-"*9}')
for i, (flor, c_km, c_hc) in enumerate(zip(novas_flores, pred_km, pred_hc)):
    print(f'  {i+1:>4} | {flor[0]:>7.1f} | {flor[1]:>7.1f} | {flor[2]:>7.1f} | {flor[3]:>7.1f} | {f"Cluster {c_km}":>9} | {f"Cluster {c_hc}":>9}')

print(f'\nNota: K-Means é o único com predict() nativo.')
print(f'Para DBSCAN e Hierárquico, novas amostras são atribuídas ao cluster mais próximo.')

In [ ]:
# Visualizar novas flores sobre os clusters K-Means (espaço PCA)
novas_pca = pca.transform(novas_flores_scaled)

plt.figure(figsize=(9, 6))

paleta_3 = ['#e74c3c', '#2ecc71', '#3498db']
for cls, color in enumerate(paleta_3):
    mask = labels_km == cls
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color,
                alpha=0.4, s=40, label=f'Cluster {cls} (treino)')

plt.scatter(cent_pca[:, 0], cent_pca[:, 1], marker='X', s=200,
            color='black', zorder=5, label='Centroides')

for i, (ponto, cluster) in enumerate(zip(novas_pca, pred_km)):
    plt.scatter(ponto[0], ponto[1], marker='*', s=250,
                color=paleta_3[cluster], edgecolors='black', linewidths=1.5,
                zorder=6, label=f'Nova flor {i+1} → Cluster {cluster}')

plt.xlabel(f'PC1 ({variancia[0]:.1%})', fontsize=11)
plt.ylabel(f'PC2 ({variancia[1]:.1%})', fontsize=11)
plt.title('K-Means — Novas Amostras Classificadas (espaço PCA)', fontsize=13, fontweight='bold')
plt.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

---
## 12. Resumo Final

In [ ]:
print('=' * 65)
print('           RESUMO — AGRUPAMENTO (CLUSTERING) IRIS')
print('=' * 65)
print()
print('  PROBLEMA')
print('  Descobrir grupos naturais SEM rótulos (não supervisionado)')
print()
print('  DADOS')
print('  150 amostras | 4 features | sem rótulos no treino')
print()
print('  PRÉ-PROCESSAMENTO')
print('  StandardScaler — obrigatório para K-Means e DBSCAN')
print()
print('  RESULTADOS')
print(f'  Algoritmo          | Clusters | Silhouette | ARI (vs real)')
print(f'  -------------------|----------|------------|-------------')
sil_km = silhouette_score(X_scaled, labels_km)
sil_hc = silhouette_score(X_scaled, labels_hc)
sil_db = silhouette_score(X_db_valid, labels_db_valid) if len(set(labels_db_valid)) > 1 else float('nan')
print(f'  K-Means            |    3     |  {sil_km:.4f}  |   {ari_km:.4f}')
print(f'  DBSCAN             |  {n_clusters_db} (+ruído)|  {sil_db:.4f}  |   {ari_db:.4f}')
print(f'  Hierárquico (Ward) |    3     |  {sil_hc:.4f}  |   {ari_hc:.4f}')
print()
print('  CARACTERÍSTICAS')
print('  K-Means     → rápido, escalável, exige k pré-definido')
print('  DBSCAN      → detecta outliers, clusters arbitrários, exige ε')
print('  Hierárquico → dendrograma interpretável, sem k fixo no treino')
print()
print('  OBSERVAÇÃO')
print('  K-Means e Hierárquico conseguiram recuperar bem as 3 espécies')
print('  setosa é completamente separada — versicolor/virginica se sobrepõem')
print('=' * 65)

---
## Próximos Passos

1. **PCA como pré-processamento** — reduzir dimensionalidade antes de clusterizar
2. **Gaussian Mixture Models (GMM)** — clusters com distribuições gaussianas sobrepostas
3. **HDBSCAN** — versão hierárquica e mais robusta do DBSCAN
4. **t-SNE / UMAP** — visualização de alta dimensionalidade

```python
# Exemplo: K-Means com PCA como pré-passo
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('kmeans', KMeans(n_clusters=3, random_state=42))
])
pipe.fit(X)
```